In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

import joblib

In [ ]:
normal_df = pd.read_csv("../data/processed/cubesat_telemetry_normal.csv")

print("Shape:", normal_df.shape)
normal_df.head()

Shape: (10796, 23)


,sequence,timestamp,position_x,position_y,position_z,roll,pitch,yaw,quaternion_q1,quaternion_q2,...,gyro_y,gyro_z,battery_voltage,battery_current,solar_power,temperature,signal_strength,packet_loss,label,fault_type
0,A,0.0000,-0.0,0.0,1.0,45.117479,44.882023,45.166314,0.192013,0.461661,...,-7.078027,9.918970,8.124377,2.193649,21.270104,25.026652,-65.595860,0.627253,0,none
1,A,0.0167,-0.0,0.0,1.0,45.234264,44.763820,45.331961,0.192685,0.461381,...,-7.085766,9.898929,8.016801,2.483561,21.108419,24.444433,-70.179468,0.815919,0,none
2,A,0.0334,-0.0,0.0,1.0,45.350321,44.645359,45.496938,0.193356,0.461100,...,-7.100504,9.858264,8.160036,1.722095,20.806295,24.482154,-73.640849,0.366242,0,none
3,A,0.0501,-0.0,0.0,1.0,45.465648,44.526663,45.661227,0.194026,0.460819,...,-7.113890,9.817497,8.175245,2.291634,20.110384,24.436773,-68.823690,0.390909,0,none
4,A,0.0668,-0.0,0.0,1.0,45.580239,44.407755,45.824842,0.194696,0.460536,...,-7.127687,9.777959,7.943917,2.014027,19.213435,24.526197,-70.532964,0.458227,0,none


In [3]:
print(normal_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10796 entries, 0 to 10795
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   sequence         10796 non-null  object 
 1   timestamp        10796 non-null  float64
 2   position_x       10796 non-null  float64
 3   position_y       10796 non-null  float64
 4   position_z       10796 non-null  float64
 5   roll             10796 non-null  float64
 6   pitch            10796 non-null  float64
 7   yaw              10796 non-null  float64
 8   quaternion_q1    10796 non-null  float64
 9   quaternion_q2    10796 non-null  float64
 10  quaternion_q3    10796 non-null  float64
 11  quaternion_w     10796 non-null  float64
 12  gyro_x           10796 non-null  float64
 13  gyro_y           10796 non-null  float64
 14  gyro_z           10796 non-null  float64
 15  battery_voltage  10796 non-null  float64
 16  battery_current  10796 non-null  float64
 17  solar_power 

In [4]:
print("\nMissing values:")
print(normal_df.isnull().sum())


Missing values:
sequence           0
timestamp          0
position_x         0
position_y         0
position_z         0
roll               0
pitch              0
yaw                0
quaternion_q1      0
quaternion_q2      0
quaternion_q3      0
quaternion_w       0
gyro_x             0
gyro_y             0
gyro_z             0
battery_voltage    0
battery_current    0
solar_power        0
temperature        0
signal_strength    0
packet_loss        0
label              0
fault_type         0
dtype: int64


In [5]:
print("\nLabels:")
print(normal_df["label"].value_counts())

print("\nFault types:")
print(normal_df["fault_type"].value_counts())


Labels:
label
0    10796
Name: count, dtype: int64

Fault types:
fault_type
none    10796
Name: count, dtype: int64


In [6]:
GYRO_FEATURES = ["gyro_x", "gyro_y", "gyro_z"]

print(normal_df[GYRO_FEATURES].describe())

             gyro_x        gyro_y        gyro_z
count  10796.000000  10796.000000  10796.000000
mean      -0.799707     -0.569887      5.194460
std      146.741458      8.009528    146.736367
min    -5383.113038    -14.970023  -5381.750721
25%       -1.401126     -7.509300      6.492736
50%        3.648699     -0.506264      7.763857
75%        8.621773      4.463740     11.208554
max       17.285904     14.970208     19.960077


In [7]:
print(
    normal_df.loc[
        normal_df[GYRO_FEATURES].abs().gt(100).any(axis=1),
        GYRO_FEATURES
    ]
)

            gyro_x     gyro_y       gyro_z
7706  -5381.914319  -1.773814 -5381.558571
7707  -5383.113038  13.195478 -5380.359327
8724  -5381.946499  -5.321866 -5381.526206
8725  -5382.202030   9.648252 -5381.270497
9742  -5382.039382  -8.869502 -5381.434747
9743  -5381.901920   6.100197 -5381.570554
10760 -5381.722103 -12.417355 -5381.750721
10761 -5381.751918   2.552745 -5381.720974


In [8]:
bad_gyro_mask = (
    normal_df[GYRO_FEATURES].abs().gt(100).any(axis=1)
)

print("Bad rows:", bad_gyro_mask.sum())

normal_clean = normal_df[~bad_gyro_mask].copy()

print("Original rows:", len(normal_df))
print("Clean rows:", len(normal_clean))
print("Removed:", len(normal_df) - len(normal_clean))

Bad rows: 8
Original rows: 10796
Clean rows: 10788
Removed: 8


In [9]:
normal_clean[GYRO_FEATURES].describe()

,gyro_x,gyro_y,gyro_z
count,10788.000000,10788.000000,10788.000000
mean,3.190856,-0.570599,9.188968
std,7.536222,8.009133,3.576240
min,-17.286007,-14.970023,4.987990
25%,-1.388050,-7.509300,6.495565
50%,3.653704,-0.506264,7.767801
75%,8.623716,4.463410,11.214471
max,17.285904,14.970208,19.960077


In [10]:
normal_clean.loc[
    normal_clean[GYRO_FEATURES].abs().gt(100).any(axis=1),
    GYRO_FEATURES
]

,gyro_x,gyro_y,gyro_z


In [ ]:
test_df = pd.read_csv("../data/processed/cubesat_telemetry_test.csv")

print("Shape:", test_df.shape)

print("\nFault distribution:")
print(test_df["fault_type"].value_counts())

print("\nLabel distribution:")
print(test_df["label"].value_counts())

Shape: (10796, 22)

Fault distribution:
fault_type
none             9595
thermal           301
power             300
communication     300
attitude          300
Name: count, dtype: int64

Label distribution:
label
0    9595
1    1201
Name: count, dtype: int64


In [12]:
bad_test_gyro = (
    test_df[GYRO_FEATURES].abs().gt(100).any(axis=1)
)

print("Extreme gyro rows:", bad_test_gyro.sum())

print(
    test_df.loc[
        bad_test_gyro,
        ["fault_type", "gyro_x", "gyro_y", "gyro_z"]
    ]
)

Extreme gyro rows: 7
      fault_type        gyro_x     gyro_y        gyro_z
897         none    -16.864197   1.445730 -21537.344352
3057       power    -16.862822   1.444699 -21537.345962
4775        none      1.416660  -4.234343 -21551.038162
7707        none -10773.957243  11.421583 -10767.957599
8725        none -10772.219320   4.326356 -10769.695974
9743        none -10771.663883  -2.769286 -10770.253264
10761       none -10770.736159  -9.864151 -10771.180527


In [13]:
bad_normal_test = (
    (test_df["fault_type"] == "none") &
    (test_df[GYRO_FEATURES].abs().gt(100).any(axis=1))
)

print("Normal artifact rows:", bad_normal_test.sum())

Normal artifact rows: 6


In [14]:
test_clean = test_df[~bad_normal_test].copy()

print("Original test rows:", len(test_df))
print("Clean test rows:", len(test_clean))
print("Removed:", len(test_df) - len(test_clean))

Original test rows: 10796
Clean test rows: 10790
Removed: 6


In [15]:
test_clean.loc[
    test_clean[GYRO_FEATURES].abs().gt(100).any(axis=1),
    ["fault_type", "gyro_x", "gyro_y", "gyro_z"]
]

,fault_type,gyro_x,gyro_y,gyro_z
3057,power,-16.862822,1.444699,-21537.345962


In [16]:
print(test_clean["fault_type"].value_counts())

fault_type
none             9589
thermal           301
power             300
communication     300
attitude          300
Name: count, dtype: int64


In [17]:
FEATURES = [
    "battery_voltage",
    "battery_current",
    "solar_power",
    "temperature",
    "signal_strength",
    "packet_loss",
    "gyro_x",
    "gyro_y",
    "gyro_z"
]

X_train = normal_clean[FEATURES]

print("Training shape:", X_train.shape)

Training shape: (10788, 9)


In [18]:
scaler_v2 = StandardScaler()

X_train_scaled = scaler_v2.fit_transform(X_train)

print("Scaled shape:", X_train_scaled.shape)

Scaled shape: (10788, 9)


In [19]:
model_v2 = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

model_v2.fit(X_train_scaled)

print("Isolation Forest V2 trained successfully.")

Isolation Forest V2 trained successfully.


In [20]:
joblib.dump(model_v2, "../models_v2/isolation_forest_v2.pkl")
joblib.dump(scaler_v2, "../models_v2/scaler_v2.pkl")

print("V2 model and scaler saved.")

V2 model and scaler saved.


In [21]:
X_test = test_clean[FEATURES]

print("Test shape:", X_test.shape)

Test shape: (10790, 9)


In [22]:
X_test_scaled = scaler_v2.transform(X_test)

print("Scaled test shape:", X_test_scaled.shape)

Scaled test shape: (10790, 9)


In [23]:
test_predictions = model_v2.predict(X_test_scaled)

test_clean["predicted_anomaly"] = (
    test_predictions == -1
).astype(int)

In [24]:
print(test_clean["predicted_anomaly"].value_counts())

predicted_anomaly
0    8335
1    2455
Name: count, dtype: int64


In [25]:
test_clean["anomaly_score"] = model_v2.decision_function(
    X_test_scaled
)

In [26]:
test_clean[[
    "fault_type",
    "label",
    "predicted_anomaly",
    "anomaly_score"
]].head(20)

,fault_type,label,predicted_anomaly,anomaly_score
0,none,0,0,0.065674
1,none,0,1,-0.001142
2,none,0,0,0.065792
3,none,0,0,0.070926
4,none,0,0,0.064026
5,none,0,0,0.058190
6,none,0,0,0.077428
7,none,0,0,0.074237
8,none,0,0,0.080623
9,none,0,0,0.103278


In [27]:
pd.crosstab(
    test_clean["fault_type"],
    test_clean["predicted_anomaly"]
)

predicted_anomaly,0,1
fault_type,,
attitude,0,300
communication,35,265
none,8178,1411
power,2,298
thermal,120,181


In [28]:
pd.crosstab(
    test_clean["fault_type"],
    test_clean["predicted_anomaly"],
    margins=True
)

predicted_anomaly,0,1,All
fault_type,,,
attitude,0,300,300
communication,35,265,300
none,8178,1411,9589
power,2,298,300
thermal,120,181,301
All,8335,2455,10790


In [29]:
print("NORMAL")
print(
    test_clean[test_clean["fault_type"] == "none"]
    ["anomaly_score"]
    .describe()
)

print("\nTHERMAL")
print(
    test_clean[test_clean["fault_type"] == "thermal"]
    ["anomaly_score"]
    .describe()
)

print("\nPOWER")
print(
    test_clean[test_clean["fault_type"] == "power"]
    ["anomaly_score"]
    .describe()
)

print("\nCOMMUNICATION")
print(
    test_clean[test_clean["fault_type"] == "communication"]
    ["anomaly_score"]
    .describe()
)

print("\nATTITUDE")
print(
    test_clean[test_clean["fault_type"] == "attitude"]
    ["anomaly_score"]
    .describe()
)

NORMAL
count    9589.000000
mean        0.038807
std         0.036554
min        -0.115404
25%         0.015766
50%         0.042875
75%         0.065231
max         0.119436
Name: anomaly_score, dtype: float64

THERMAL
count    301.000000
mean      -0.009036
std        0.024649
min       -0.088591
25%       -0.024472
50%       -0.005184
75%        0.008265
max        0.053011
Name: anomaly_score, dtype: float64

POWER
count    300.000000
mean      -0.101448
std        0.029409
min       -0.162247
25%       -0.120877
50%       -0.103740
75%       -0.087719
max        0.008810
Name: anomaly_score, dtype: float64

COMMUNICATION
count    300.000000
mean      -0.038658
std        0.030612
min       -0.111681
25%       -0.057701
50%       -0.040813
75%       -0.025553
max        0.105381
Name: anomaly_score, dtype: float64

ATTITUDE
count    300.000000
mean      -0.060606
std        0.022872
min       -0.125451
25%       -0.075777
50%       -0.059517
75%       -0.043014
max       -0.002934


In [30]:
thresholds = [0, -0.01, -0.02, -0.03, -0.04, -0.05, -0.06]

for threshold in thresholds:

    predictions = (
        test_clean["anomaly_score"] < threshold
    ).astype(int)

    normal = test_clean[test_clean["fault_type"] == "none"]
    faults = test_clean[test_clean["label"] == 1]

    false_positives = (
        normal["anomaly_score"] < threshold
    ).sum()

    detected_faults = (
        faults["anomaly_score"] < threshold
    ).sum()

    print(
        f"Threshold {threshold:>5}: "
        f"False positives = {false_positives:4}, "
        f"Faults detected = {detected_faults:4}/1201"
    )

Threshold     0: False positives = 1411, Faults detected = 1044/1201
Threshold -0.01: False positives =  980, Faults detected =  967/1201
Threshold -0.02: False positives =  665, Faults detected =  910/1201
Threshold -0.03: False positives =  445, Faults detected =  838/1201
Threshold -0.04: False positives =  264, Faults detected =  710/1201
Threshold -0.05: False positives =  157, Faults detected =  605/1201
Threshold -0.06: False positives =   79, Faults detected =  504/1201


In [31]:
threshold = -0.02

test_clean["threshold_prediction"] = (
    test_clean["anomaly_score"] < threshold
).astype(int)

pd.crosstab(
    test_clean["fault_type"],
    test_clean["threshold_prediction"]
)

threshold_prediction,0,1
fault_type,,
attitude,9,291
communication,60,240
none,8924,665
power,7,293
thermal,215,86


In [32]:
threshold = -0.01

test_clean["threshold_prediction"] = (
    test_clean["anomaly_score"] < threshold
).astype(int)

pd.crosstab(
    test_clean["fault_type"],
    test_clean["threshold_prediction"]
)

threshold_prediction,0,1
fault_type,,
attitude,3,297
communication,45,255
none,8609,980
power,5,295
thermal,181,120


In [34]:
thermal = test_clean[test_clean["fault_type"] == "thermal"]

print(
    thermal[[
        "temperature",
        "anomaly_score",
        "threshold_prediction"
    ]].sort_values("temperature").tail(30)
)

      temperature  anomaly_score  threshold_prediction
1271         52.1      -0.002561                     0
1272         52.2      -0.001963                     0
1273         52.3      -0.023195                     1
1274         52.4       0.011398                     0
1275         52.5       0.002719                     0
1276         52.6       0.011770                     0
1277         52.7      -0.033419                     1
1278         52.8      -0.032192                     1
1279         52.9       0.015983                     0
1280         53.0       0.002357                     0
1281         53.1      -0.014154                     1
1282         53.2      -0.073540                     1
1283         53.3      -0.017102                     1
1284         53.4       0.012522                     0
1285         53.5       0.004450                     0
1286         53.6       0.008254                     0
1287         53.7      -0.014971                     1
1288      

In [36]:
print(normal_clean.columns.tolist())

['sequence', 'timestamp', 'position_x', 'position_y', 'position_z', 'roll', 'pitch', 'yaw', 'quaternion_q1', 'quaternion_q2', 'quaternion_q3', 'quaternion_w', 'gyro_x', 'gyro_y', 'gyro_z', 'battery_voltage', 'battery_current', 'solar_power', 'temperature', 'signal_strength', 'packet_loss', 'label', 'fault_type']


In [37]:
print(test_clean.columns.tolist())

['timestamp', 'position_x', 'position_y', 'position_z', 'roll', 'pitch', 'yaw', 'quaternion_q1', 'quaternion_q2', 'quaternion_q3', 'quaternion_w', 'gyro_x', 'gyro_y', 'gyro_z', 'battery_voltage', 'battery_current', 'solar_power', 'temperature', 'signal_strength', 'packet_loss', 'label', 'fault_type', 'predicted_anomaly', 'anomaly_score', 'threshold_prediction']


In [38]:
WINDOW = 20

normal_clean["temperature_rolling_mean"] = (
    normal_clean["temperature"]
    .rolling(window=WINDOW, min_periods=1)
    .mean()
)

test_clean["temperature_rolling_mean"] = (
    test_clean["temperature"]
    .rolling(window=WINDOW, min_periods=1)
    .mean()
)

In [39]:
print(normal_clean[
    ["temperature", "temperature_rolling_mean"]
].head(25))

    temperature  temperature_rolling_mean
0     25.026652                 25.026652
1     24.444433                 24.735542
2     24.482154                 24.651079
3     24.436773                 24.597503
4     24.526197                 24.583242
5     23.735555                 24.441960
6     24.741270                 24.484719
7     24.640793                 24.504228
8     24.242203                 24.475114
9     25.155046                 24.543108
10    25.239889                 24.606451
11    24.443353                 24.592860
12    24.855866                 24.613091
13    27.158624                 24.794915
14    27.120433                 24.949949
15    23.945910                 24.887197
16    24.085552                 24.840041
17    23.940597                 24.790072
18    26.057054                 24.856755
19    24.601471                 24.843991
20    25.239674                 24.854642
21    26.996967                 24.982269
22    24.989535                 25

In [40]:
thermal = test_clean[test_clean["fault_type"] == "thermal"]

print(
    thermal[
        ["temperature", "temperature_rolling_mean"]
    ].tail(20)
)

      temperature  temperature_rolling_mean
1281         53.1                     52.15
1282         53.2                     52.25
1283         53.3                     52.35
1284         53.4                     52.45
1285         53.5                     52.55
1286         53.6                     52.65
1287         53.7                     52.75
1288         53.8                     52.85
1289         53.9                     52.95
1290         54.0                     53.05
1291         54.1                     53.15
1292         54.2                     53.25
1293         54.3                     53.35
1294         54.4                     53.45
1295         54.5                     53.55
1296         54.6                     53.65
1297         54.7                     53.75
1298         54.8                     53.85
1299         54.9                     53.95
1300         55.0                     54.05


In [41]:
FEATURES = [
    "battery_voltage",
    "battery_current",
    "solar_power",
    "temperature",
    "temperature_rolling_mean",
    "signal_strength",
    "packet_loss",
    "gyro_x",
    "gyro_y",
    "gyro_z"
]

In [42]:
X_train = normal_clean[FEATURES]

print("Training shape:", X_train.shape)

Training shape: (10788, 10)


In [43]:
scaler_v21 = StandardScaler()

X_train_scaled = scaler_v21.fit_transform(X_train)

print("Scaled training shape:", X_train_scaled.shape)

Scaled training shape: (10788, 10)


In [44]:
model_v21 = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

model_v21.fit(X_train_scaled)

print("Isolation Forest V2.1 trained successfully!")

Isolation Forest V2.1 trained successfully!


In [45]:
joblib.dump(
    model_v21,
    "../models_v21/isolation_forest_v21.pkl"
)

joblib.dump(
    scaler_v21,
    "../models_v21/scaler_v21.pkl"
)

print("V2.1 model and scaler saved.")

V2.1 model and scaler saved.


In [46]:
X_test_v21 = test_clean[FEATURES]

print("Test shape:", X_test_v21.shape)

Test shape: (10790, 10)


In [47]:
X_test_v21_scaled = scaler_v21.transform(X_test_v21)

print("Scaled test shape:", X_test_v21_scaled.shape)

Scaled test shape: (10790, 10)


In [48]:
predictions_v21 = model_v21.predict(X_test_v21_scaled)

test_clean["predicted_anomaly_v21"] = (
    predictions_v21 == -1
).astype(int)

In [49]:
test_clean["anomaly_score_v21"] = (
    model_v21.decision_function(X_test_v21_scaled)
)

In [50]:
print(
    test_clean["predicted_anomaly_v21"].value_counts()
)

predicted_anomaly_v21
0    8544
1    2246
Name: count, dtype: int64


In [51]:
pd.crosstab(
    test_clean["fault_type"],
    test_clean["predicted_anomaly_v21"]
)

predicted_anomaly_v21,0,1
fault_type,,
attitude,4,296
communication,56,244
none,8468,1121
power,5,295
thermal,11,290


In [52]:
print("NORMAL")
print(
    test_clean[test_clean["fault_type"] == "none"]
    ["anomaly_score_v21"]
    .describe()
)

print("\nTHERMAL")
print(
    test_clean[test_clean["fault_type"] == "thermal"]
    ["anomaly_score_v21"]
    .describe()
)

NORMAL
count    9589.000000
mean        0.042888
std         0.034740
min        -0.104307
25%         0.020621
50%         0.046039
75%         0.067922
max         0.125039
Name: anomaly_score_v21, dtype: float64

THERMAL
count    301.000000
mean      -0.042610
std        0.024987
min       -0.119917
25%       -0.055866
50%       -0.042296
75%       -0.029384
max        0.055642
Name: anomaly_score_v21, dtype: float64


In [53]:
thresholds = [0, -0.01, -0.02, -0.03]

for threshold in thresholds:

    predictions = (
        test_clean["anomaly_score_v21"] < threshold
    ).astype(int)

    normal = test_clean[test_clean["fault_type"] == "none"]
    faults = test_clean[test_clean["label"] == 1]

    false_positives = (
        normal["anomaly_score_v21"] < threshold
    ).sum()

    detected_faults = (
        faults["anomaly_score_v21"] < threshold
    ).sum()

    print(
        f"Threshold {threshold:>5}: "
        f"False positives = {false_positives:4}, "
        f"Faults detected = {detected_faults:4}/1201"
    )

Threshold     0: False positives = 1121, Faults detected = 1125/1201
Threshold -0.01: False positives =  744, Faults detected = 1081/1201
Threshold -0.02: False positives =  466, Faults detected =  987/1201
Threshold -0.03: False positives =  276, Faults detected =  852/1201


In [54]:
threshold = -0.01

test_clean["final_prediction"] = (
    test_clean["anomaly_score_v21"] < threshold
).astype(int)

pd.crosstab(
    test_clean["fault_type"],
    test_clean["final_prediction"]
)

final_prediction,0,1
fault_type,,
attitude,12,288
communication,90,210
none,8845,744
power,6,294
thermal,12,289


In [55]:
comm = test_clean[
    test_clean["fault_type"] == "communication"
]

normal = test_clean[
    test_clean["fault_type"] == "none"
]

print("NORMAL COMMUNICATION")
print(
    normal[["signal_strength", "packet_loss"]].describe()
)

print("\nCOMMUNICATION FAULT")
print(
    comm[["signal_strength", "packet_loss"]].describe()
)

NORMAL COMMUNICATION
       signal_strength  packet_loss
count      9589.000000  9589.000000
mean        -69.942148     0.501430
std           3.008355     0.198688
min         -80.463410     0.000000
25%         -72.014771     0.365952
50%         -69.965179     0.502174
75%         -67.890521     0.635381
max         -59.041019     1.246557

COMMUNICATION FAULT
       signal_strength  packet_loss
count       300.000000   300.000000
mean        -82.500000    15.250000
std           7.253073     8.558627
min         -95.000000     0.500000
25%         -88.750000     7.875000
50%         -82.500000    15.250000
75%         -76.250000    22.625000
max         -70.000000    30.000000


In [56]:
missed_comm = test_clean[
    (test_clean["fault_type"] == "communication") &
    (test_clean["final_prediction"] == 0)
]

print(missed_comm[
    ["signal_strength", "packet_loss", "anomaly_score_v21"]
].describe())

       signal_strength  packet_loss  anomaly_score_v21
count        90.000000    90.000000          90.000000
mean        -76.801375     8.525622           0.014119
std           6.951342     8.202584           0.023658
min         -95.000000     0.500000          -0.009922
25%         -78.507525     2.892559          -0.004578
50%         -74.138796     5.383779           0.005597
75%         -72.027592    10.538880           0.028888
max         -70.000000    30.000000           0.105175


In [57]:
WINDOW = 20

normal_clean["signal_strength_rolling_mean"] = (
    normal_clean["signal_strength"]
    .rolling(window=WINDOW, min_periods=1)
    .mean()
)

normal_clean["packet_loss_rolling_mean"] = (
    normal_clean["packet_loss"]
    .rolling(window=WINDOW, min_periods=1)
    .mean()
)

test_clean["signal_strength_rolling_mean"] = (
    test_clean["signal_strength"]
    .rolling(window=WINDOW, min_periods=1)
    .mean()
)

test_clean["packet_loss_rolling_mean"] = (
    test_clean["packet_loss"]
    .rolling(window=WINDOW, min_periods=1)
    .mean()
)

In [58]:
comm = test_clean[
    test_clean["fault_type"] == "communication"
]

print(
    comm[
        [
            "signal_strength",
            "signal_strength_rolling_mean",
            "packet_loss",
            "packet_loss_rolling_mean"
        ]
    ].tail(20)
)

      signal_strength  signal_strength_rolling_mean  packet_loss  \
5280       -93.411371                    -92.617057    28.125418   
5281       -93.494983                    -92.700669    28.224080   
5282       -93.578595                    -92.784281    28.322742   
5283       -93.662207                    -92.867893    28.421405   
5284       -93.745819                    -92.951505    28.520067   
5285       -93.829431                    -93.035117    28.618729   
5286       -93.913043                    -93.118729    28.717391   
5287       -93.996656                    -93.202341    28.816054   
5288       -94.080268                    -93.285953    28.914716   
5289       -94.163880                    -93.369565    29.013378   
5290       -94.247492                    -93.453177    29.112040   
5291       -94.331104                    -93.536789    29.210702   
5292       -94.414716                    -93.620401    29.309365   
5293       -94.498328                    -93.704

In [59]:
FEATURES_V22 = [
    "battery_voltage",
    "battery_current",
    "solar_power",

    "temperature",
    "temperature_rolling_mean",

    "signal_strength",
    "signal_strength_rolling_mean",

    "packet_loss",
    "packet_loss_rolling_mean",

    "gyro_x",
    "gyro_y",
    "gyro_z"
]

X_train_v22 = normal_clean[FEATURES_V22]

print("Training shape:", X_train_v22.shape)

Training shape: (10788, 12)


In [60]:
scaler_v22 = StandardScaler()

X_train_v22_scaled = scaler_v22.fit_transform(X_train_v22)

print("Scaled shape:", X_train_v22_scaled.shape)

Scaled shape: (10788, 12)


In [61]:
model_v22 = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

model_v22.fit(X_train_v22_scaled)

print("Isolation Forest V2.2 trained successfully!")

Isolation Forest V2.2 trained successfully!


In [62]:
joblib.dump(
    model_v22,
    "../models_v22/isolation_forest_v22.pkl"
)

joblib.dump(
    scaler_v22,
    "../models_v22/scaler_v22.pkl"
)

print("V2.2 model and scaler saved.")

V2.2 model and scaler saved.


In [63]:
X_test_v22 = test_clean[FEATURES_V22]

print("Test shape:", X_test_v22.shape)

Test shape: (10790, 12)


In [64]:
X_test_v22_scaled = scaler_v22.transform(X_test_v22)

print("Scaled test shape:", X_test_v22_scaled.shape)

Scaled test shape: (10790, 12)


In [65]:
predictions_v22 = model_v22.predict(X_test_v22_scaled)

test_clean["predicted_anomaly_v22"] = (
    predictions_v22 == -1
).astype(int)

In [66]:
test_clean["anomaly_score_v22"] = (
    model_v22.decision_function(X_test_v22_scaled)
)

In [67]:
print(
    test_clean["predicted_anomaly_v22"].value_counts()
)
pd.crosstab(
    test_clean["fault_type"],
    test_clean["predicted_anomaly_v22"]
)

predicted_anomaly_v22
0    8707
1    2083
Name: count, dtype: int64


predicted_anomaly_v22,0,1
fault_type,,
attitude,6,294
communication,18,282
none,8625,964
power,9,291
thermal,49,252


In [68]:
thresholds = [0, -0.01, -0.02, -0.03]

for threshold in thresholds:

    predictions = (
        test_clean["anomaly_score_v22"] < threshold
    ).astype(int)

    normal = test_clean[test_clean["fault_type"] == "none"]
    faults = test_clean[test_clean["label"] == 1]

    false_positives = (
        normal["anomaly_score_v22"] < threshold
    ).sum()

    detected_faults = (
        faults["anomaly_score_v22"] < threshold
    ).sum()

    print(
        f"Threshold {threshold:>5}: "
        f"False positives = {false_positives:4}, "
        f"Faults detected = {detected_faults:4}/1201"
    )

Threshold     0: False positives =  964, Faults detected = 1119/1201
Threshold -0.01: False positives =  605, Faults detected = 1055/1201
Threshold -0.02: False positives =  375, Faults detected =  955/1201
Threshold -0.03: False positives =  197, Faults detected =  810/1201


In [69]:
threshold = -0.01

test_clean["final_prediction_v22"] = (
    test_clean["anomaly_score_v22"] < threshold
).astype(int)

pd.crosstab(
    test_clean["fault_type"],
    test_clean["final_prediction_v22"]
)

final_prediction_v22,0,1
fault_type,,
attitude,14,286
communication,23,277
none,8984,605
power,11,289
thermal,98,203
